# test

## Requirements

In [1]:
import numpy as np
import deeplake
import os
import joblib
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, balanced_accuracy_score, confusion_matrix 
from sklearn.linear_model import SGDClassifier
from sklearn.pipeline import make_pipeline
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.interpolate import interp1d

import random
from datetime import datetime
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
import joblib


LABEL_MAP = {
    "Stroma": 0,
    "Normal": 1,
    "G3":     2,
    "G4":     3,
    "G5":     4
}

REVERSE_LABEL_MAP = {0: "Stroma",
    1: "Normal", 
    2: "G3",
    3: "G4",
    4: "G5"
}

## Amplitudes extraction

In [2]:
def Flat_log_Fourier(src_image, L=0.01):
    
    # Compute FFT of the source image
    fft_src = np.fft.fft2(src_image, axes=(-2, -1))
    amp_src = np.abs(fft_src)
    phase_src = np.angle(fft_src)

    # Shift amplitude spectra to center the low-frequency components
    amp_src = np.fft.fftshift(amp_src, axes=(-2, -1))

    # Defining the square of low amplitudes to be swapped
    _, height, width = amp_src.shape
    radius = int(np.floor(min(height, width) * L))
    center_h = height // 2
    center_w = width // 2

    h_start, h_end = center_h - radius, center_h + radius + 1
    w_start, w_end = center_w - radius, center_w + radius + 1

    small_amp = amp_src[:, h_start:h_end, w_start:w_end]

    return np.log(small_amp.reshape(-1))

dataset_path_akoya_1 = f"/home/leolr-int/nfs/data/data/patched/dim_256/Train/Subset3_Train_1_Akoya"
akoya_1 = deeplake.open_read_only(dataset_path_akoya_1)

# Preprocessing
src_img = akoya_1[200]["patch"].transpose((2, 0, 1)) 
print(Flat_log_Fourier(src_img))
print(Flat_log_Fourier(src_img).shape)

[10.72968828 10.87027149 10.32206744 11.36520953 11.94543284 11.85280867
 10.99149009 11.06714864  9.80301996 11.3958142  11.94771888 11.31831403
 16.43901608 11.31831403 11.94771888 11.3958142   9.80301996 11.06714864
 10.99149009 11.85280867 11.94543284 11.36520953 10.32206744 10.87027149
 10.72968828 10.23796142 11.28155811 10.79568398 11.70807848 12.0833677
 11.75780574 10.95748935 11.62618943 10.98913147 12.27952787 11.41043939
 10.36950676 16.07781091 10.36950676 11.41043939 12.27952787 10.98913147
 11.62618943 10.95748935 11.75780574 12.0833677  11.70807848 10.79568398
 11.28155811 10.23796142  8.01799344 10.16877095  9.08203862 10.9730164
 11.37710229 10.66138433 10.32184102 10.72468183 10.84279119 11.36165889
 10.75955693  9.99530605 16.42815754  9.99530605 10.75955693 11.36165889
 10.84279119 10.72468183 10.32184102 10.66138433 11.37710229 10.9730164
  9.08203862 10.16877095  8.01799344]
(75,)


## Amplitudes dataset creation

In [3]:
# specify 'group' (WSI) to avoid data leakage!



def _process_fourier(img_L):
    """Helper pour le multiprocessing."""
    img, L = img_L
    return Flat_log_Fourier(img, L)

def extract_features_parallel(imgs, L, max_workers=4):
    """Extrait les features en parallèle pour une liste d’images."""
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        return list(executor.map(_process_fourier, [(img, L) for img in imgs]))

def load_data(input_root, train_or_test, L=0.01, max_workers=4, binary=False):
    X, y, groups = [], [], []

    for scanner in list_train_scanners:
        for i in range(1, 27):  
            dataset_name = f"Subset3_{train_or_test}_{i}_{scanner}"
            dataset_path = f"{input_root}/{train_or_test}/{dataset_name}"
            if not os.path.exists(dataset_path):
                continue

            print(f"Loading {dataset_path}")
            dataset = deeplake.open_read_only(dataset_path)

            imgs, labels, group_list = [], [], []

            for sample in tqdm(dataset, desc=dataset_name):
                patch = sample["patch"]
                if patch.shape != (256, 256, 3):
                    continue

                img = patch.transpose(2, 0, 1)
                label = int(sample["label"]) #modify here for binary classification

                imgs.append(img)
                labels.append(label)
                group_list.append(dataset_name)

            if imgs:
                features = extract_features_parallel(imgs, L, max_workers=4)
                X.extend(features)
                y.extend(labels)
                groups.extend(group_list)

    return np.array(X), np.array(y), np.array(groups)




## Add the training of your Machine Learning Model

In [4]:
# blank

## Compute the distributions of amplitudes per label (non binary)

In [5]:
# non binary distributions per coordinate
# for each label, we compute the dsitribution of each coordinate inside the amplitude vector. 

def compute_smoothed_cdfs(X_train, y_train, num_points=512):
    labels = np.unique(y_train)
    n_features = X_train.shape[1]
    cdf_dict = {}

    for label in tqdm(labels, desc="Processing labels"):
        X_label = X_train[y_train == label]
        cdf_dict[label] = {}

        for feature_idx in range(n_features):
            values = X_label[:, feature_idx]

            # Fit KDE
            kde = gaussian_kde(values)

            # Evaluate KDE on linspace covering the data range
            xmin, xmax = np.percentile(values, [0.5, 99.5])  # robust range
            xs = np.linspace(xmin, xmax, num_points)
            pdf = kde(xs)

            # Normalize to CDF
            cdf = np.cumsum(pdf)
            cdf /= cdf[-1]  # Normalize to [0,1]

            cdf_dict[label][feature_idx] = (xs, cdf)

    return cdf_dict

## Test with everything

In [6]:
list_train_scanners = ['Akoya', 'Leica']
input_root = '/home/leolr-int/nfs/data/data/patched/dim_256'
train_or_test = 'Train'
X_train, y_train, groups_train = load_data(input_root, train_or_test, L=0.01)
X_test, y_test, groups_test = load_data(input_root, 'Test', L=0.01)


Loading /home/leolr-int/nfs/data/data/patched/dim_256/Train/Subset3_Train_1_Akoya


Subset3_Train_1_Akoya: 100%|███████████████████████████████████████████████████████████████████| 10675/10675 [00:43<00:00, 248.16it/s]
/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/deeplake/__init__.py:322: UserWarning: Global variable 'akoya_1' of type <class 'deeplake._deeplake.ReadOnlyDataset'> may cause issues when using fork-based multiprocessing. Consider avoiding global variables of this type, or pass to subprocess as an agrument or by manual pickling.
  warnings.warn(
/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/deeplake/__init__.py:322: UserWarning: Global variable 'akoya_1' of type <class 'deeplake._deeplake.ReadOnlyDataset'> may cause issues when using fork-based multiprocessing. Consider avoiding global variables of this type, or pass to subprocess as an agrument or by manual pickling.
  warnings.warn(


Loading /home/leolr-int/nfs/data/data/patched/dim_256/Train/Subset3_Train_2_Akoya


Subset3_Train_2_Akoya: 100%|███████████████████████████████████████████████████████████████████| 11743/11743 [00:37<00:00, 311.83it/s]


Loading /home/leolr-int/nfs/data/data/patched/dim_256/Train/Subset3_Train_3_Akoya


...

Subset3_Test_10_Leica: 100%|█████████████████████████████████████████████████████████████████████| 1513/1513 [00:05<00:00, 258.20it/s]


Loading /home/leolr-int/nfs/data/data/patched/dim_256/Test/Subset3_Test_11_Leica


Subset3_Test_11_Leica: 100%|█████████████████████████████████████████████████████████████████████| 5107/5107 [00:28<00:00, 178.10it/s]


Loading /home/leolr-int/nfs/data/data/patched/dim_256/Test/Subset3_Test_12_Leica


Subset3_Test_12_Leica: 100%|█████████████████████████████████████████████████████████████████████| 6908/6908 [00:38<00:00, 178.78it/s]


In [7]:
path='/home/leolr-int/nfs/transformed_data/weights'
os.makedirs(f"{path}/{train_or_test}", exist_ok=True)
np.save(f'{path}/{train_or_test}/amplitudes.npy', X_train)
np.save(f'{path}/{train_or_test}/labels.npy', y_train)
np.save(f'{path}/{train_or_test}/groups.npy', groups_train)


In [10]:
akoya_id = int(X_test.shape[0]/2)
print(akoya_id)
X_train_akoya = X_train[:akoya_id,]
y_train_akoya = y_train[:akoya_id]
print(X_train_akoya.shape)
print(y_train_akoya.shape)

distrib_Akoya = compute_smoothed_cdfs(X_train_akoya, y_train_akoya)
save_path = "/home/leolr-int/nfs/transformed_data/weights/cdf_Akoya.pkl"
joblib.dump(distrib_Akoya, save_path)

82777
(82777, 75)
(82777,)


Processing labels: 100%|████████████████████████████████████████████████████████████████████████████████| 5/5 [00:29<00:00,  5.93s/it]


['/home/leolr-int/nfs/transformed_data/weights/cdf_Akoya.pkl']